# P4 HWPX 690 KoE5 Embedding + Retrieval Quick Check

This notebook checks file size and retrieval quality for the P4 HWPX 690 corpus on Colab/GCV L4.

```text
1. Environment check
   - Google Drive mount
   - requirements.txt install
   - CUDA/GPU label

2. P4 HWPX 690 corpus load
   - outputs/parsing_p4_hwpx_690/chunks_v1_690.jsonl  # R0 clean text baseline
   - outputs/parsing_p4_hwpx_690/chunks_v2_690.jsonl  # structured corpus

3. Chroma load
   - v1: clean text chunks
   - v2: text/table/fact_candidates chunks, toc excluded

4. Retrieval quick check
   - R0: v1 + KoE5 dense top5
   - R1: v2 + KoE5 dense top30 -> reranker top5
   - R2: v2 + KoE5 dense top30 + BM25 top30 -> RRF top30 -> reranker top5

5. Evaluation
   - eval_batch_01.csv ~ eval_batch_25.csv only
   - default EVAL_SAMPLE_SIZE=100
   - append results to outputs/retrieval_eval_p4_hwpx/experiment_logs/retrieval_experiments.csv
```

Notes:

- Do not store Chroma DB on Google Drive. This notebook uses local runtime storage by default.
- Do not create `.npy` embedding cache unless there is a specific reason.
- Original RFP files, source evidence, Chroma DB, and embedding cache are not GitHub upload targets.


In [1]:
# ===== Install dependencies in the active kernel =====
# VS Code connected to Colab must install packages inside the Colab runtime.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_project_root_for_install() -> Path:
    candidates = [
        Path(os.environ.get('RFP_PROJECT_ROOT', '')) if os.environ.get('RFP_PROJECT_ROOT') else None,
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/Colab Notebooks/DB_RAG_Codeit_Project'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for base in candidates:
        if base and (base / 'requirements.txt').exists():
            return base.resolve()

    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print('Google Drive mount skipped:', exc)

    search_roots = [Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), Path('/content/drive')]
    matches = []
    for root in search_roots:
        if root.exists():
            try:
                matches.extend(root.rglob('requirements.txt'))
            except Exception as exc:
                print(f'skip search root: {root} | {exc}')
    matches = sorted(set(matches), key=lambda p: (0 if 'DB_RAG_Codeit_Project' in str(p) else 1, len(str(p))))
    if matches:
        print('found requirements candidates:')
        for i, p in enumerate(matches[:10]):
            print(f'  [{i}] {p}')
        return matches[0].parent.resolve()

    raise FileNotFoundError('requirements.txt was not found under /content/drive. Check PROJECT_ROOT_OVERRIDE or mount Drive.')


PROJECT_ROOT_FOR_INSTALL = find_project_root_for_install()
REQUIREMENTS_PATH = PROJECT_ROOT_FOR_INSTALL / 'requirements.txt'
print('PROJECT_ROOT_FOR_INSTALL:', PROJECT_ROOT_FOR_INSTALL)
print('REQUIREMENTS_PATH:', REQUIREMENTS_PATH)

REQUIRED_IMPORTS = {
    'chromadb': 'chromadb',
    'rank-bm25': 'rank_bm25',
    'sentence-transformers': 'sentence_transformers',
    'transformers': 'transformers',
    'openpyxl': 'openpyxl',
}
missing = [name for name, module in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]

if missing:
    print('missing packages:', missing)
    print('installing:', REQUIREMENTS_PATH)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS_PATH)])
    print('dependency install complete. If imports still fail, restart the Colab runtime once and rerun from the top.')
else:
    print('required packages already installed')


Mounted at /content/drive
found requirements candidates:
  [0] /content/drive/MyDrive/DB_RAG_Codeit_Project/requirements.txt
  [1] /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/eval/evaluation/requirements.txt
PROJECT_ROOT_FOR_INSTALL: /content/drive/MyDrive/DB_RAG_Codeit_Project
REQUIREMENTS_PATH: /content/drive/MyDrive/DB_RAG_Codeit_Project/requirements.txt
missing packages: ['chromadb', 'rank-bm25']
installing: /content/drive/MyDrive/DB_RAG_Codeit_Project/requirements.txt
dependency install complete. If imports still fail, restart the Colab runtime once and rerun from the top.


In [2]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


python: 3.12.13
torch: 2.10.0+cu128
cuda_available: True
gpu: NVIDIA L4


In [3]:
# ===== User parameters =====
# Quick check default: P4 HWPX 690 corpus + 100 eval questions.

CORPUS_LIMIT = 690
EVAL_SAMPLE_SIZE = 100
EVAL_SAMPLE_SEED = 42

CORPUS_NAME = f"p4_hwpx_{CORPUS_LIMIT}"
CORPUS_COVERAGE = f"p4_hwpx_full_{CORPUS_LIMIT}"
CORPUS_VERSION_V1 = "v1_clean_text"
CORPUS_VERSION_V2 = "v2_hwpx_table_aware"

EMBEDDING_MODEL = "nlpai-lab/KoE5"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

EMBEDDING_BATCH_SIZE = 64
CHROMA_ADD_BATCH_SIZE = 512
RERANK_BATCH_SIZE = 32

FINAL_TOP_K = 5
DENSE_TOP_K = 30
BM25_TOP_K = 30
RRF_K = 60

RUN_RERANKER_EXPERIMENTS = True
RECREATE_CHROMA_COLLECTIONS = True
SAVE_EMBEDDING_CACHE = False  # Keep this False to avoid large .npy embedding cache files.

ENABLE_COVERAGE_GATE = True
MIN_TOP1_SCORE_FOR_WEAK_SIGNAL = 0.20
UNVERIFIED_ANSWER = "\ud655\uc778\ub418\uc9c0 \uc54a\uc2b5\ub2c8\ub2e4."

# Google Drive / Colab default. Override with RFP_PROJECT_ROOT if needed.
PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/DB_RAG_Codeit_Project"
# GCV VM example. Uncomment on GCV and comment out the Google Drive line above.
# PROJECT_ROOT_OVERRIDE = "/home/USER/DB_RAG_Codeit_Project"

# Keep Chroma outside Google Drive to avoid slow sync and heavy cache issues.
# predictions/csv stay under OUTPUT_DIR on Drive; Chroma uses local runtime storage.
LOCAL_CHROMA_ROOT_OVERRIDE = "/content/chroma_retrieval_eval_p4_hwpx"
# GCV VM example:
# LOCAL_CHROMA_ROOT_OVERRIDE = "/tmp/chroma_retrieval_eval_p4_hwpx"


def _project_root_score(path: Path) -> tuple:
    text = str(path).replace('\\', '/')
    return (
        0 if 'DB_RAG_Codeit_Project' in text else 1,
        0 if (path / 'outputs' / f'parsing_p4_hwpx_{CORPUS_LIMIT}').exists() else 1,
        len(text),
    )


def detect_project_root() -> Path:
    candidates = []
    env_root = os.environ.get('RFP_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    candidates.extend([
        Path.cwd(),
        *Path.cwd().parents,
        Path('/content/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/Colab Notebooks/DB_RAG_Codeit_Project'),
        Path(r'C:/Users/yoosy/Desktop/codeit/project_2nd'),
    ])
    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'outputs' / f'parsing_p4_hwpx_{CORPUS_LIMIT}' / f'chunks_v2_{CORPUS_LIMIT}.jsonl').exists():
            return base.resolve()
        if (base / 'notebooks' / 'rag').exists() and (base / 'requirements.txt').exists():
            return base.resolve()

    search_roots = [Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), Path('/content/drive')]
    matches = []
    target = Path('outputs') / f'parsing_p4_hwpx_{CORPUS_LIMIT}' / f'chunks_v2_{CORPUS_LIMIT}.jsonl'
    for root in search_roots:
        if root.exists():
            try:
                matches.extend(root.rglob(str(target)))
            except Exception as exc:
                print(f'skip search root: {root} | {exc}')
    matches = sorted(set(matches), key=lambda p: _project_root_score(p.parents[2]))
    if matches:
        print('found P4 chunks candidates:')
        for i, p in enumerate(matches[:10]):
            print(f'  [{i}] {p}')
        return matches[0].parents[2].resolve()

    return Path.cwd().resolve()


def gpu_label() -> str:
    if not torch.cuda.is_available():
        return "CPU"
    name = torch.cuda.get_device_name(0)
    if "L4" in name.upper():
        return "L4"
    if "4060" in name:
        return "RTX4060"
    return "CUDA"

PROJECT_ROOT = detect_project_root()
P4_DIR = PROJECT_ROOT / "outputs" / f"parsing_p4_hwpx_{CORPUS_LIMIT}"
V1_CHUNKS_PATH = P4_DIR / f"chunks_v1_{CORPUS_LIMIT}.jsonl"
V2_CHUNKS_PATH = P4_DIR / f"chunks_v2_{CORPUS_LIMIT}.jsonl"
EVAL_DIR = PROJECT_ROOT / "data" / "eval"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "retrieval_eval_p4_hwpx"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
if LOCAL_CHROMA_ROOT_OVERRIDE:
    CHROMA_DIR = Path(LOCAL_CHROMA_ROOT_OVERRIDE) / f"p4_hwpx_{CORPUS_LIMIT}"
else:
    CHROMA_DIR = OUTPUT_DIR / "chroma"
EXTENDED_EVAL_SCRIPT = PROJECT_ROOT / "notebooks" / "eval" / "evaluation" / "scripts" / "run_retrieval_eval_extended.py"

EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBEDDING_DEVICE_LABEL = gpu_label()

for path in [OUTPUT_DIR, PREDICTION_DIR, CHROMA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMBEDDING_DEVICE:", EMBEDDING_DEVICE)
print("EMBEDDING_DEVICE_LABEL:", EMBEDDING_DEVICE_LABEL)
print("V1_CHUNKS_PATH exists:", V1_CHUNKS_PATH.exists(), V1_CHUNKS_PATH)
print("V2_CHUNKS_PATH exists:", V2_CHUNKS_PATH.exists(), V2_CHUNKS_PATH)
print("EVAL_DIR exists:", EVAL_DIR.exists(), EVAL_DIR)
print("EXTENDED_EVAL_SCRIPT exists:", EXTENDED_EVAL_SCRIPT.exists(), EXTENDED_EVAL_SCRIPT)
print("CHROMA_DIR local:", CHROMA_DIR)


PROJECT_ROOT: /content/drive/MyDrive/DB_RAG_Codeit_Project
EMBEDDING_DEVICE: cuda
EMBEDDING_DEVICE_LABEL: L4
V1_CHUNKS_PATH exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_690/chunks_v1_690.jsonl
V2_CHUNKS_PATH exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_690/chunks_v2_690.jsonl
EVAL_DIR exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/data/eval
EXTENDED_EVAL_SCRIPT exists: True /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/eval/evaluation/scripts/run_retrieval_eval_extended.py
CHROMA_DIR local: /content/chroma_retrieval_eval_p4_hwpx/p4_hwpx_690


In [4]:
def file_mib(path: Path) -> float:
    return round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0.0

file_check_df = pd.DataFrame([
    {"file": "chunks_v1", "path": str(V1_CHUNKS_PATH), "exists": V1_CHUNKS_PATH.exists(), "mib": file_mib(V1_CHUNKS_PATH)},
    {"file": "chunks_v2", "path": str(V2_CHUNKS_PATH), "exists": V2_CHUNKS_PATH.exists(), "mib": file_mib(V2_CHUNKS_PATH)},
])
display(file_check_df)

if not V1_CHUNKS_PATH.exists() or not V2_CHUNKS_PATH.exists():
    raise FileNotFoundError("P4 HWPX chunks_v1/chunks_v2 files are required before running retrieval quick check.")


,file,path,exists,mib
0,chunks_v1,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_690/chunks_v1_690.jsonl,True,189.18
1,chunks_v2,/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_690/chunks_v2_690.jsonl,True,251.44


In [5]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_chunks(path: Path) -> pd.DataFrame:
    df = pd.DataFrame(read_jsonl(path))
    if "content" not in df.columns:
        raise ValueError("chunks JSONL must include content")
    if "embed_enabled" not in df.columns:
        df["embed_enabled"] = True
    if "chunk_type" not in df.columns:
        df["chunk_type"] = "text"
    df = df.copy()
    df["content"] = df["content"].fillna("").astype(str)
    return df


def select_embedding_chunks(df: pd.DataFrame) -> pd.DataFrame:
    selected = df[df["embed_enabled"].fillna(True).astype(bool)].copy()
    selected = selected[selected["chunk_type"].astype(str).ne("toc")].copy()
    selected = selected[selected["content"].str.strip().ne("")].copy()
    selected = selected.drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)
    return selected

chunks_v1_raw = load_chunks(V1_CHUNKS_PATH)
chunks_v2_raw = load_chunks(V2_CHUNKS_PATH)
chunks_v1 = select_embedding_chunks(chunks_v1_raw)
chunks_v2 = select_embedding_chunks(chunks_v2_raw)

summary = pd.DataFrame([
    {"version": "v1", "raw_rows": len(chunks_v1_raw), "selected_rows": len(chunks_v1), "unique_docs": chunks_v1["doc_key"].nunique(), "duplicate_chunk_id": int(chunks_v1["chunk_id"].duplicated().sum()), "chunk_types": dict(chunks_v1_raw["chunk_type"].value_counts())},
    {"version": "v2", "raw_rows": len(chunks_v2_raw), "selected_rows": len(chunks_v2), "unique_docs": chunks_v2["doc_key"].nunique(), "duplicate_chunk_id": int(chunks_v2["chunk_id"].duplicated().sum()), "chunk_types": dict(chunks_v2_raw["chunk_type"].value_counts())},
])
display(summary)
display(chunks_v1[["chunk_id", "source_file", "chunk_type", "content"]].head(2))
display(chunks_v2[["chunk_id", "source_file", "chunk_type", "content"]].head(2))


,version,raw_rows,selected_rows,unique_docs,duplicate_chunk_id,chunk_types
0,v1,75912,75638,685,0,"{'text': 75638, 'toc': 274}"
1,v2,124740,124472,685,0,"{'table': 97278, 'text': 26506, 'fact_candidates': 691, 'toc': 265}"


,chunk_id,source_file,chunk_type,content
0,doc_c6761a6fb2b7_text_text_0001_part_001_9972157211c2,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n-1-\n제 안 요 청 서\n고려대학교\n차세대 포털·학사 정보시스템 구축 사업
1,doc_c6761a6fb2b7_text_text_0002_part_001_468c8342bef7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,"[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n※ 본 자료는 제안내용의 설명을 위한 배포자료로, 이외 목적으로 무단복제, 전달 및 사용하는 행위를 금함.\n-2-"


,chunk_id,source_file,chunk_type,content
0,doc_c6761a6fb2b7_text_text_0001_part_001_9972157211c2,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n-1-\n제 안 요 청 서\n고려대학교\n차세대 포털·학사 정보시스템 구축 사업
1,doc_c6761a6fb2b7_text_text_0002_part_001_468c8342bef7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,"[문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf | 사업명: 차세대 포털·학사 정보시스템 구축사업 | 발주기관: 고려대학교 | 섹션: 문서 시작 | 유형: text]\n※ 본 자료는 제안내용의 설명을 위한 배포자료로, 이외 목적으로 무단복제, 전달 및 사용하는 행위를 금함.\n-2-"


In [6]:
def parse_list_cell(value) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
        return [str(parsed)]
    except Exception:
        if "|" in text:
            return [x.strip() for x in text.split("|") if x.strip()]
        return [text]


def load_eval_questions(eval_dir: Path, canonical_only: bool = True) -> pd.DataFrame:
    if canonical_only:
        csv_paths = [eval_dir / f"eval_batch_{idx:02d}.csv" for idx in range(1, 26)]
        missing = [str(path) for path in csv_paths if not path.exists()]
        if missing:
            raise FileNotFoundError(f"Missing canonical eval CSV files: {missing}")
    else:
        csv_paths = sorted(eval_dir.glob("eval_batch_*.csv"))
        if not csv_paths:
            raise FileNotFoundError(f"No eval CSV files under {eval_dir}")
    frames = []
    for csv_path in csv_paths:
        frame = pd.read_csv(csv_path)
        frame["source_eval_file"] = csv_path.name
        frames.append(frame)
    df = pd.concat(frames, ignore_index=True)
    required = ["id", "type", "difficulty", "question", "ground_truth_answer", "ground_truth_docs"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing eval columns: {missing}")
    df["ground_truth_doc_list"] = df["ground_truth_docs"].apply(parse_list_cell)
    df["gt_doc_count"] = df["ground_truth_doc_list"].apply(len)
    return df

eval_df_all = load_eval_questions(EVAL_DIR, canonical_only=True)
if EVAL_SAMPLE_SIZE and EVAL_SAMPLE_SIZE > 0:
    eval_df = eval_df_all.sample(n=min(EVAL_SAMPLE_SIZE, len(eval_df_all)), random_state=EVAL_SAMPLE_SEED).sort_values(["source_eval_file", "id"]).reset_index(drop=True)
else:
    eval_df = eval_df_all.reset_index(drop=True)

print("eval all rows:", len(eval_df_all))
print("eval selected rows:", len(eval_df))
display(eval_df["gt_doc_count"].value_counts().sort_index().rename_axis("gt_doc_count").reset_index(name="rows"))
display(eval_df[["source_eval_file", "id", "question", "gt_doc_count", "ground_truth_doc_list"]].head(5))


eval all rows: 500
eval selected rows: 100


,gt_doc_count,rows
0,1,61
1,2,31
2,3,7
3,4,1


,source_eval_file,id,question,gt_doc_count,ground_truth_doc_list
0,eval_batch_01.csv,Q001,한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?,1,[한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]
1,eval_batch_01.csv,Q003,코이카(KOICA) 전자조달의 '우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 및 지역의회 연계 개선 PMC 용역' 예산 규모는 어떻게 됩니까?,1,[KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp]
2,eval_batch_01.csv,Q010,국립중앙의료원의 차세대 응급의료 상황관리시스템 사업과 한국원자력연구원의 선량평가시스템 고도화 사업 모두 시스템 고도화 및 구축을 목적으로 합니다. 두 기관 중 어느 곳의 예산이 더 많이 배정되었나요?,2,"[국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp]"
3,eval_batch_01.csv,Q012,나노종합기술원의 스마트 팹 서비스 관련 설비온라인 사업과 부산국제영화제의 BIFF & ACFM 온라인서비스 재개발 사업의 추진 목적 및 예상되는 기대효과를 대비해서 요약해 주세요.,2,"[나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp, (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]"
4,eval_batch_01.csv,Q016,"그렇다면 1단계와 2단계로 진행되는 이 용수공급사업의 전체 예산 중에서, 약 30%를 올해 기초 조사 및 기반 설계에 사용한다고 가정할 때 해당 금액은 총 얼마입니까?",1,[한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp]


In [7]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=EMBEDDING_DEVICE)
print("loaded embedding model:", EMBEDDING_MODEL)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

loaded embedding model: nlpai-lab/KoE5


In [8]:
def prefixed_passages(texts: list[str]) -> list[str]:
    return [f"passage: {text}" for text in texts]


def prefixed_queries(texts: list[str]) -> list[str]:
    return [f"query: {text}" for text in texts]


def encode_passages(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    return embedding_model.encode(prefixed_passages(texts), batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)


def encode_queries(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    return embedding_model.encode(prefixed_queries(texts), batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)


def metadata_value(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    if isinstance(value, (list, tuple, set)):
        return " > ".join(map(str, value))
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)


def make_chroma_metadata(row: pd.Series) -> dict[str, str]:
    base = {}
    meta = row.get("metadata", {})
    if isinstance(meta, dict):
        base.update(meta)
    source_ref = row.get("source_ref", {})
    if isinstance(source_ref, dict):
        for key in ["source_store_id", "block_id", "part_index", "content_hash"]:
            if key in source_ref:
                base[key] = source_ref.get(key, "")
    for key in ["doc_id", "doc_key", "source_file", "source_format", "chunk_id", "chunk_type", "fact_type", "fact_status", "fact_confidence"]:
        if key in row.index:
            base[key] = row.get(key, "")
    return {key: metadata_value(value) for key, value in base.items()}


def reset_collection(client, collection_name: str):
    if RECREATE_CHROMA_COLLECTIONS:
        try:
            client.delete_collection(collection_name)
            print("deleted existing collection:", collection_name)
        except Exception:
            pass
    return client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})


def build_or_load_collection_streaming(df: pd.DataFrame, collection_name: str):
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = reset_collection(client, collection_name)
    if collection.count() == len(df) and not RECREATE_CHROMA_COLLECTIONS:
        print("reuse existing collection:", collection_name, collection.count())
        return collection
    ids = df["chunk_id"].astype(str).tolist()
    docs = df["content"].astype(str).tolist()
    total = len(ids)
    for start in tqdm(range(0, total, CHROMA_ADD_BATCH_SIZE), desc=f"encode+chroma add {collection_name}"):
        end = min(start + CHROMA_ADD_BATCH_SIZE, total)
        batch_df = df.iloc[start:end]
        batch_docs = docs[start:end]
        batch_embeddings = encode_passages(batch_docs, batch_size=EMBEDDING_BATCH_SIZE)
        batch_metadatas = [make_chroma_metadata(row) for _, row in batch_df.iterrows()]
        collection.add(ids=ids[start:end], documents=batch_docs, embeddings=batch_embeddings.tolist(), metadatas=batch_metadatas)
    print("collection count:", collection_name, collection.count())
    return collection

collection_v1_name = f"rfp_p4_hwpx_{CORPUS_LIMIT}_v1_koe5"
collection_v2_name = f"rfp_p4_hwpx_{CORPUS_LIMIT}_v2_koe5"
collection_v1 = build_or_load_collection_streaming(chunks_v1, collection_v1_name)
collection_v2 = build_or_load_collection_streaming(chunks_v2, collection_v2_name)


encode+chroma add rfp_p4_hwpx_690_v1_koe5:   0%|          | 0/148 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

collection count: rfp_p4_hwpx_690_v2_koe5 124472


In [9]:
def normalize_doc_name(value: str) -> str:
    text = str(value or "").strip().lower()
    text = re.sub(r"\.(hwp|hwpx|pdf|json)$", "", text)
    text = re.sub(r"\s+", "", text)
    return text


def result_from_chroma_item(rank: int, doc: str, metadata: dict, distance: float, embedding=None) -> dict[str, Any]:
    score = 1.0 - float(distance) if distance is not None else math.nan
    return {"rank": rank, "filename": metadata.get("source_file", ""), "doc_id": metadata.get("doc_id", ""), "chunk_id": metadata.get("chunk_id", ""), "score": score, "text": doc, "metadata": metadata, "_embedding": embedding}


def dense_candidates(collection, question: str, n_results: int) -> tuple[list[dict[str, Any]], np.ndarray, float]:
    t0 = time.perf_counter()
    q_emb = encode_queries([question])[0]
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=n_results, include=["documents", "metadatas", "distances"])
    docs = res.get("documents", [[]])[0]
    metadatas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]
    embs = []
    items = []
    for idx, (doc, meta, dist) in enumerate(zip(docs, metadatas, distances), start=1):
        emb = embs[idx - 1] if len(embs) >= idx else None
        items.append(result_from_chroma_item(idx, doc, meta or {}, dist, emb))
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return items, q_emb, elapsed_ms


def dedupe_by_doc(candidates: list[dict[str, Any]], k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    selected = []
    seen = set()
    for item in candidates:
        doc_key = normalize_doc_name(item.get("filename") or item.get("doc_id"))
        if not doc_key:
            doc_key = str(item.get("doc_id") or item.get("chunk_id"))
        if doc_key in seen:
            continue
        seen.add(doc_key)
        item = dict(item)
        item["rank"] = len(selected) + 1
        selected.append(item)
        if len(selected) >= k:
            break
    return selected


def bm25_tokenize(text: str) -> list[str]:
    terms = re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())
    tokens = []
    for term in terms:
        tokens.append(term)
        if re.search(r"[가-힣]", term) and len(term) >= 2:
            tokens.extend(term[i:i+2] for i in range(len(term)-1))
    return tokens


class BM25Index:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        tokenized = [bm25_tokenize(text) for text in tqdm(self.df["content"].astype(str).tolist(), desc="bm25 tokenize")]
        self.index = BM25Okapi(tokenized)

    def search(self, question: str, top_k: int) -> list[dict[str, Any]]:
        scores = self.index.get_scores(bm25_tokenize(question))
        order = np.argsort(scores)[::-1][:top_k]
        items = []
        for rank, idx in enumerate(order, start=1):
            row = self.df.iloc[int(idx)]
            items.append({"rank": rank, "filename": row.get("source_file", ""), "doc_id": row.get("doc_id", ""), "chunk_id": row.get("chunk_id", ""), "score": float(scores[idx]), "text": row.get("content", ""), "metadata": make_chroma_metadata(row)})
        return items


def rrf_fuse(result_lists: list[list[dict[str, Any]]], rrf_k: int = RRF_K, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    by_chunk = {}
    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            chunk_id = item.get("chunk_id")
            if not chunk_id:
                continue
            if chunk_id not in by_chunk:
                by_chunk[chunk_id] = dict(item)
                by_chunk[chunk_id]["rrf_score"] = 0.0
            by_chunk[chunk_id]["rrf_score"] += 1.0 / (rrf_k + rank)
    fused = sorted(by_chunk.values(), key=lambda item: item.get("rrf_score", 0.0), reverse=True)[:top_k]
    for rank, item in enumerate(fused, start=1):
        item["rank"] = rank
        item["score"] = float(item.get("rrf_score", item.get("score", 0.0)))
    return fused


def clean_contexts_for_json(items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    clean = []
    for rank, item in enumerate(items, start=1):
        clean.append({"rank": rank, "filename": item.get("filename", ""), "doc_id": item.get("doc_id", ""), "chunk_id": item.get("chunk_id", ""), "score": float(item.get("score", 0.0)) if item.get("score") is not None else None, "text": item.get("text", ""), "metadata": item.get("metadata", {})})
    return clean


def context_signal_summary(contexts: list[dict[str, Any]]) -> dict[str, Any]:
    clean_contexts = clean_contexts_for_json(contexts)
    unique_docs = {normalize_doc_name(item.get("filename") or item.get("doc_id")) for item in clean_contexts if item.get("filename") or item.get("doc_id")}
    top1_score = clean_contexts[0].get("score") if clean_contexts else None
    return {"top1_score": top1_score, "top5_doc_count": len(unique_docs)}


def infer_retrieval_status(signal: dict[str, Any]) -> tuple[str, str]:
    if signal.get("top5_doc_count", 0) == 0:
        return "insufficient_context", "no_retrieved_context"
    top1_score = signal.get("top1_score")
    if ENABLE_COVERAGE_GATE and top1_score is not None and top1_score < MIN_TOP1_SCORE_FOR_WEAK_SIGNAL:
        return "weak_signal", "top1_score_below_threshold"
    return "sufficient_context", "retrieval_signal_passed"

bm25_v2 = BM25Index(chunks_v2)


bm25 tokenize:   0%|          | 0/124472 [00:00<?, ?it/s]

In [10]:
reranker = None
if RUN_RERANKER_EXPERIMENTS:
    reranker = CrossEncoder(RERANKER_MODEL, device=EMBEDDING_DEVICE)
    print("loaded reranker:", RERANKER_MODEL)
else:
    print("reranker disabled")


def rerank(question: str, candidates: list[dict[str, Any]], top_k: int = FINAL_TOP_K) -> tuple[list[dict[str, Any]], float]:
    if reranker is None:
        return candidates[:top_k], 0.0
    if not candidates:
        return [], 0.0
    t0 = time.perf_counter()
    pairs = [(question, item.get("text", "")) for item in candidates]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    reranked = []
    for item, score in zip(candidates, scores):
        updated = dict(item)
        updated["score"] = float(score)
        updated["rerank_score"] = float(score)
        reranked.append(updated)
    reranked = sorted(reranked, key=lambda item: item.get("rerank_score", 0.0), reverse=True)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    for rank, item in enumerate(reranked[:top_k], start=1):
        item["rank"] = rank
    return reranked[:top_k], elapsed_ms


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

loaded reranker: BAAI/bge-reranker-v2-m3


In [11]:
@dataclass
class RetrievalExperiment:
    experiment_id: str
    corpus_name: str
    corpus_version: str
    retrieval_method: str
    collection: Any
    chunks_df: pd.DataFrame
    bm25: Any = None
    collection_name: str = ""
    dense_top_k: int = DENSE_TOP_K
    bm25_top_k: int = 0
    fetch_k: int = FINAL_TOP_K
    rrf_k: int = RRF_K
    reranker_model: str = ""
    rerank_top_k: int = 0
    final_top_k: int = FINAL_TOP_K
    use_hybrid: bool = False
    use_reranker: bool = False

experiments = [
    RetrievalExperiment(f"R0_p4_hwpx_{CORPUS_LIMIT}_v1_dense_top5", CORPUS_NAME, CORPUS_VERSION_V1, "dense", collection_v1, chunks_v1, collection_name=collection_v1_name, dense_top_k=FINAL_TOP_K, fetch_k=FINAL_TOP_K),
    RetrievalExperiment(f"R1_p4_hwpx_{CORPUS_LIMIT}_v2_dense_rerank_top5", CORPUS_NAME, CORPUS_VERSION_V2, "dense_rerank", collection_v2, chunks_v2, bm25=bm25_v2, collection_name=collection_v2_name, dense_top_k=DENSE_TOP_K, fetch_k=DENSE_TOP_K, use_reranker=True, reranker_model=RERANKER_MODEL, rerank_top_k=FINAL_TOP_K),
    RetrievalExperiment(f"R2_p4_hwpx_{CORPUS_LIMIT}_v2_hybrid_rrf_rerank_top5", CORPUS_NAME, CORPUS_VERSION_V2, "hybrid_rrf_rerank", collection_v2, chunks_v2, bm25=bm25_v2, collection_name=collection_v2_name, dense_top_k=DENSE_TOP_K, bm25_top_k=BM25_TOP_K, fetch_k=DENSE_TOP_K, use_hybrid=True, use_reranker=True, reranker_model=RERANKER_MODEL, rerank_top_k=FINAL_TOP_K),
]
if not RUN_RERANKER_EXPERIMENTS:
    experiments = [exp for exp in experiments if not exp.use_reranker]

pd.DataFrame([{"experiment_id": exp.experiment_id, "corpus_version": exp.corpus_version, "retrieval_method": exp.retrieval_method, "collection": exp.collection_name, "chunks": len(exp.chunks_df), "dense_top_k": exp.dense_top_k, "bm25_top_k": exp.bm25_top_k, "reranker": exp.use_reranker} for exp in experiments])


,experiment_id,corpus_version,retrieval_method,collection,chunks,dense_top_k,bm25_top_k,reranker
0,R0_p4_hwpx_690_v1_dense_top5,v1_clean_text,dense,rfp_p4_hwpx_690_v1_koe5,75638,5,0,False
1,R1_p4_hwpx_690_v2_dense_rerank_top5,v2_hwpx_table_aware,dense_rerank,rfp_p4_hwpx_690_v2_koe5,124472,30,0,True
2,R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5,v2_hwpx_table_aware,hybrid_rrf_rerank,rfp_p4_hwpx_690_v2_koe5,124472,30,30,True


In [12]:
def retrieve_one(question: str, exp: RetrievalExperiment) -> tuple[list[dict[str, Any]], float, float, float]:
    retrieval_start = time.perf_counter()
    rerank_ms = 0.0
    dense_items, q_emb, dense_ms = dense_candidates(exp.collection, question, n_results=exp.fetch_k)
    if exp.use_hybrid:
        bm25_items = exp.bm25.search(question, top_k=exp.bm25_top_k) if exp.bm25 is not None else []
        rrf_top_k = exp.fetch_k if exp.use_reranker else exp.final_top_k
        candidates = rrf_fuse([dense_items, bm25_items], rrf_k=exp.rrf_k, top_k=rrf_top_k)
    else:
        candidates = dense_items[:exp.fetch_k]
    if exp.use_reranker:
        candidates, rerank_ms = rerank(question, candidates, top_k=exp.final_top_k)
    final_contexts = dedupe_by_doc(candidates, k=exp.final_top_k)
    total_ms = (time.perf_counter() - retrieval_start) * 1000
    retrieval_ms = total_ms - rerank_ms
    return final_contexts, retrieval_ms, rerank_ms, total_ms


def retriever_config(exp: RetrievalExperiment) -> dict[str, Any]:
    return {
        "experiment_id": exp.experiment_id, "corpus_name": exp.corpus_name, "corpus_version": exp.corpus_version, "corpus_coverage": CORPUS_COVERAGE,
        "embedding_model": EMBEDDING_MODEL, "embedding_device": EMBEDDING_DEVICE_LABEL, "vector_db": "chroma", "collection_name": exp.collection_name,
        "retrieval_method": exp.retrieval_method, "retriever_type": exp.retrieval_method, "dense_top_k": exp.dense_top_k, "bm25_top_k": exp.bm25_top_k,
        "fetch_k": exp.fetch_k, "rrf_k": exp.rrf_k if exp.use_hybrid else "", "reranker": bool(exp.use_reranker), "reranker_model": exp.reranker_model,
        "rerank_top_k": exp.rerank_top_k, "final_top_k": exp.final_top_k, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
        "toc_policy": "excluded_from_embedding", "metadata_boost": False, "doc_dedup": True, "coverage_gate_enabled": ENABLE_COVERAGE_GATE,
        "min_top1_score_for_weak_signal": MIN_TOP1_SCORE_FOR_WEAK_SIGNAL, "unverified_answer": UNVERIFIED_ANSWER,
        "eval_sample_size": EVAL_SAMPLE_SIZE, "eval_sample_seed": EVAL_SAMPLE_SEED, "save_embedding_cache": SAVE_EMBEDDING_CACHE,
    }


def write_predictions(exp: RetrievalExperiment, eval_df: pd.DataFrame) -> Path:
    pred_path = PREDICTION_DIR / f"{exp.experiment_id}_predictions.jsonl"
    config = retriever_config(exp)
    with pred_path.open("w", encoding="utf-8") as f:
        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=exp.experiment_id):
            contexts, retrieval_ms, rerank_ms, total_ms = retrieve_one(row["question"], exp)
            signal = context_signal_summary(contexts)
            retrieval_status, coverage_reason = infer_retrieval_status(signal)
            record = {"id": row["id"], "source_eval_file": row.get("source_eval_file", ""), "question": row["question"], "answer": "", "retrieved_contexts": clean_contexts_for_json(contexts), "corpus_coverage": CORPUS_COVERAGE, "retrieval_status": retrieval_status, "coverage_reason": coverage_reason, "top1_score": signal["top1_score"], "top5_doc_count": signal["top5_doc_count"], "latency_ms": total_ms, "retrieval_ms": retrieval_ms, "rerank_ms": rerank_ms, "model_name": "retrieval_only", "embedding_model": EMBEDDING_MODEL, "retriever_config": config}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return pred_path

prediction_paths = []
for exp in experiments:
    path = write_predictions(exp, eval_df)
    prediction_paths.append((exp, path))
    print(exp.experiment_id, path)


R0_p4_hwpx_690_v1_dense_top5:   0%|          | 0/100 [00:00<?, ?it/s]

R0_p4_hwpx_690_v1_dense_top5 /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/predictions/R0_p4_hwpx_690_v1_dense_top5_predictions.jsonl


R1_p4_hwpx_690_v2_dense_rerank_top5:   0%|          | 0/100 [00:00<?, ?it/s]

R1_p4_hwpx_690_v2_dense_rerank_top5 /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/predictions/R1_p4_hwpx_690_v2_dense_rerank_top5_predictions.jsonl


R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5:   0%|          | 0/100 [00:00<?, ?it/s]

R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5 /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/predictions/R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5_predictions.jsonl


In [13]:
def run_extended_eval(exp: RetrievalExperiment, pred_path: Path):
    if not EXTENDED_EVAL_SCRIPT.exists():
        raise FileNotFoundError(EXTENDED_EVAL_SCRIPT)
    cmd = [sys.executable, str(EXTENDED_EVAL_SCRIPT), "--project-root", str(PROJECT_ROOT), "--eval-dir", str(EVAL_DIR), "--predictions", str(pred_path), "--output-dir", str(OUTPUT_DIR), "--prediction-ids-only", "--top-k", str(exp.final_top_k), "--experiment-id", exp.experiment_id, "--experiment-name", exp.retrieval_method, "--eval-scope", "sample" if EVAL_SAMPLE_SIZE else "canonical_01_25", "--notes", f"P4 HWPX quickcheck; corpus={exp.corpus_name}/{exp.corpus_version}; eval_batch_01_25; sample_size={EVAL_SAMPLE_SIZE}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(" ".join(cmd))
    print(result.stdout[-2500:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"extended eval failed: {exp.experiment_id}")

for exp, pred_path in prediction_paths:
    run_extended_eval(exp, pred_path)

summary_log = OUTPUT_DIR / "experiment_logs" / "retrieval_experiments.csv"
summary_df = pd.read_csv(summary_log)
current = summary_df[summary_df["experiment_id"].isin([exp.experiment_id for exp in experiments])].tail(len(experiments))
metric_cols = ["experiment_id", "num_eval_questions", "hit_at_5_any", "mrr_at_5", "ndcg_at_5", "doc_recall_at_5", "single_doc_mrr_at_5", "multi_doc_ndcg_at_5", "retrieval_ms_p95", "rerank_ms_avg", "total_retrieval_ms_avg"]
display(current[metric_cols])


/usr/bin/python3 /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/eval/evaluation/scripts/run_retrieval_eval_extended.py --project-root /content/drive/MyDrive/DB_RAG_Codeit_Project --eval-dir /content/drive/MyDrive/DB_RAG_Codeit_Project/data/eval --predictions /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/predictions/R0_p4_hwpx_690_v1_dense_top5_predictions.jsonl --output-dir /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx --prediction-ids-only --top-k 5 --experiment-id R0_p4_hwpx_690_v1_dense_top5 --experiment-name dense --eval-scope sample --notes P4 HWPX quickcheck; corpus=p4_hwpx_690/v1_clean_text; eval_batch_01_25; sample_size=100
eval_results: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/R0_p4_hwpx_690_v1_dense_top5_eval_results_extended.csv
summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/R0_p4_hwpx_690_v1_dense_top5_summary_extended.json
summary_log

,experiment_id,num_eval_questions,hit_at_5_any,mrr_at_5,ndcg_at_5,doc_recall_at_5,single_doc_mrr_at_5,multi_doc_ndcg_at_5,retrieval_ms_p95,rerank_ms_avg,total_retrieval_ms_avg
3,R0_p4_hwpx_690_v1_dense_top5,100,0.86,0.798667,0.725949,0.744167,0.749180,0.668841,25.444765,0.000000,24.543042
4,R1_p4_hwpx_690_v2_dense_rerank_top5,100,0.95,0.928333,0.852876,0.848333,0.898907,0.766533,34.163337,1623.635362,1654.778259
5,R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5,100,0.99,0.955833,0.888483,0.891667,0.927596,0.804973,7879.963771,1598.154382,5947.249682


In [14]:
# ===== Qualitative preview: 5 questions x R0/R1/R2 =====
PREVIEW_N_QUESTIONS = 5
MAX_TEXT_CHARS = 500

pred_cache = {}
for exp, pred_path in prediction_paths:
    with pred_path.open("r", encoding="utf-8") as f:
        pred_cache[exp.experiment_id] = [json.loads(line) for line in f if line.strip()]
selected_keys = []
for row in pred_cache[experiments[0].experiment_id][:PREVIEW_N_QUESTIONS]:
    selected_keys.append((row.get("source_eval_file", ""), str(row.get("id", ""))))
for source_eval_file, qid in selected_keys:
    base_row = next((r for r in pred_cache[experiments[0].experiment_id] if r.get("source_eval_file", "") == source_eval_file and str(r.get("id", "")) == qid), None)
    if not base_row:
        continue
    print("\n" + "=" * 100)
    print(f"[{source_eval_file} / {qid}] {base_row.get('question')}")
    gt = eval_df[(eval_df["source_eval_file"].astype(str) == source_eval_file) & (eval_df["id"].astype(str) == qid)]
    if not gt.empty:
        print("GT docs:", gt.iloc[0].get("ground_truth_doc_list"))
    for exp in experiments:
        rows = [r for r in pred_cache[exp.experiment_id] if r.get("source_eval_file", "") == source_eval_file and str(r.get("id", "")) == qid]
        if not rows:
            continue
        rec = rows[0]
        print("\n---", exp.experiment_id, "---")
        for ctx in rec.get("retrieved_contexts", [])[:FINAL_TOP_K]:
            text = str(ctx.get("text", "")).replace("\n", " ")
            if len(text) > MAX_TEXT_CHARS:
                text = text[:MAX_TEXT_CHARS].rstrip() + " ..."
            print(f"#{ctx.get('rank')} score={ctx.get('score'):.4f} file={ctx.get('filename')} chunk={ctx.get('chunk_id')}")
            print(text)



[eval_batch_01.csv / Q001] 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
GT docs: ['한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp']

--- R0_p4_hwpx_690_v1_dense_top5 ---
#1 score=0.7065 file=한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp chunk=doc_f1c6cddebe51_text_text_0004_part_001_35ceab976987
[문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차세대 통합정보시스템(ERP) 구축 | 발주기관: 한국가스공사 | 섹션: Ⅲ. 일반사항 | 유형: text] Ⅲ. 일반사항 1. 사 업 명 : 차세대 통합정보시스템(ERP) 구축 [표 | 섹션: Ⅲ. 일반사항 | rows: 1 | cols: 3] 컬럼: Ⅰ | col_2 | 개요 2. 사업목적 ○ 기술지원 종료(‘27년)에 대비한 ERP 업그레이드 - 제조사(SAP社)의 기술지원 종료 이후 세법 개정, 보안취약점 발생시 원활한 대응이 불가 등 기술지원 종료 대비한 선제적 대응으로 안정적 서비스 제공 ○ 정보시스템 복잡도 해소 및 접근‧활용 간소화로 업무생산성 향상 - ‘09년 도입 이후 종합적인 개선 없이 단편적 수정으로 복잡도 증가 및 대사·검증 등 수작업에 따른 비효율 해소 * 원가결산, 급여정산 등 검증을 위한 수작업 및 대용량 데이터 처리시 속도지연 발생 - 시스템 접근, 데이터 탐 ...

--- R1_p4_hwpx_690_v2_dense_rerank_top5 ---
#1 score=0.9116 file=한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp chunk=doc_f1c6cddebe51_text_text_0004_part_001_48fffb407f32
[문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차

## 해석 기준

- `R0`: 구조화하지 않은 clean text baseline입니다. v2가 의미 있는지 비교하는 기준선입니다.
- `R1`: dense 검색 결과 30개를 reranker로 재정렬합니다. 이전 실험의 R4 조건입니다.
- `R2`: dense + BM25를 RRF로 합친 뒤 reranker를 적용합니다. 이전 실험의 R6 조건입니다.
- eval은 기본 100개 sample입니다. 전체 500문항을 보려면 `EVAL_SAMPLE_SIZE = 0`으로 바꿔서 다시 실행하면 됩니다.


## Failure Analysis

To keep notebook output readable, this section uses compact summaries first and full details only through selected CASE_ID values. Full analysis rows are saved as CSV.


In [15]:
# ===== Failure Analysis: build data and compact summaries =====
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 140)


def _normalize_doc_name_for_analysis(value: str) -> str:
    text = str(value or "").strip().lower()
    text = re.sub(r"\.(hwp|hwpx|pdf|json)$", "", text)
    text = re.sub(r"\s+", "", text)
    return text


def _load_prediction_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def _safe_contexts(value):
    return value if isinstance(value, list) else []


def _context_doc_names(contexts):
    docs = []
    for ctx in _safe_contexts(contexts):
        filename = ctx.get("filename") or ctx.get("metadata", {}).get("source_file") or ctx.get("doc_id", "")
        docs.append(str(filename))
    return docs


def _context_chunk_types(contexts):
    chunk_types = []
    for ctx in _safe_contexts(contexts):
        meta = ctx.get("metadata") or {}
        chunk_types.append(str(meta.get("chunk_type", ctx.get("chunk_type", ""))))
    return chunk_types


def _hit_any_at_k(pred_docs, gt_docs, k=5):
    pred_norm = {_normalize_doc_name_for_analysis(x) for x in pred_docs[:k] if str(x).strip()}
    gt_norm = {_normalize_doc_name_for_analysis(x) for x in gt_docs if str(x).strip()}
    return int(bool(pred_norm & gt_norm))


def _all_gold_docs_hit_at_k(pred_docs, gt_docs, k=5):
    pred_norm = {_normalize_doc_name_for_analysis(x) for x in pred_docs[:k] if str(x).strip()}
    gt_norm = {_normalize_doc_name_for_analysis(x) for x in gt_docs if str(x).strip()}
    if not gt_norm:
        return 0
    return int(gt_norm.issubset(pred_norm))


def _doc_recall_at_k(pred_docs, gt_docs, k=5):
    pred_norm = {_normalize_doc_name_for_analysis(x) for x in pred_docs[:k] if str(x).strip()}
    gt_norm = {_normalize_doc_name_for_analysis(x) for x in gt_docs if str(x).strip()}
    if not gt_norm:
        return np.nan
    return len(pred_norm & gt_norm) / len(gt_norm)


def _rank_first_gold(pred_docs, gt_docs, k=5):
    gt_norm = {_normalize_doc_name_for_analysis(x) for x in gt_docs if str(x).strip()}
    for idx, doc in enumerate(pred_docs[:k], start=1):
        if _normalize_doc_name_for_analysis(doc) in gt_norm:
            return idx
    return None


def _short(value, max_chars=70):
    if isinstance(value, (list, tuple, set)):
        text = " | ".join(map(str, value))
    else:
        text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) <= max_chars else text[:max_chars].rstrip() + " ..."


def _display_title(title):
    print()
    print("=" * 100)
    print(title)

try:
    _prediction_paths = prediction_paths
except NameError:
    _prediction_paths = []

if not _prediction_paths:
    pred_dir = PREDICTION_DIR if "PREDICTION_DIR" in globals() else Path(PROJECT_ROOT) / "outputs" / "retrieval_eval_p4_hwpx" / "predictions"
    pred_files = sorted(Path(pred_dir).glob("R*_p4_hwpx_*_predictions.jsonl"))
    _prediction_paths = []
    for pred_path in pred_files:
        exp_id = pred_path.name.replace("_predictions.jsonl", "")
        dummy_exp = type("ExperimentRef", (), {"experiment_id": exp_id, "retrieval_method": exp_id})()
        _prediction_paths.append((dummy_exp, pred_path))

if not _prediction_paths:
    raise FileNotFoundError("prediction_paths is missing and no predictions JSONL files were found.")

_eval_lookup = eval_df.copy()
_eval_lookup["_source_eval_file_str"] = _eval_lookup["source_eval_file"].astype(str)
_eval_lookup["_id_str"] = _eval_lookup["id"].astype(str)

analysis_rows = []
for exp, pred_path in _prediction_paths:
    pred_df = _load_prediction_jsonl(Path(pred_path))
    for _, row in pred_df.iterrows():
        source_eval_file = str(row.get("source_eval_file", ""))
        qid = str(row.get("id", ""))
        gt_match = _eval_lookup[(_eval_lookup["_source_eval_file_str"] == source_eval_file) & (_eval_lookup["_id_str"] == qid)]
        if gt_match.empty:
            continue
        gt_row = gt_match.iloc[0]
        gt_docs = gt_row.get("ground_truth_doc_list", [])
        contexts = _safe_contexts(row.get("retrieved_contexts", []))
        pred_docs = _context_doc_names(contexts)
        chunk_types = _context_chunk_types(contexts)
        first_rank = _rank_first_gold(pred_docs, gt_docs, k=FINAL_TOP_K)
        top1_score = row.get("top1_score", None)
        if top1_score is None and contexts:
            top1_score = contexts[0].get("score")
        analysis_rows.append({
            "experiment_id": getattr(exp, "experiment_id", str(exp)),
            "retrieval_method": getattr(exp, "retrieval_method", ""),
            "source_eval_file": source_eval_file,
            "id": qid,
            "question": row.get("question", ""),
            "type": gt_row.get("type", ""),
            "difficulty": gt_row.get("difficulty", ""),
            "gt_doc_count": len(gt_docs),
            "hit_at_5": _hit_any_at_k(pred_docs, gt_docs, FINAL_TOP_K),
            "all_gold_docs_hit_at_5": _all_gold_docs_hit_at_k(pred_docs, gt_docs, FINAL_TOP_K),
            "doc_recall_at_5": _doc_recall_at_k(pred_docs, gt_docs, FINAL_TOP_K),
            "first_gold_rank": first_rank,
            "gt_docs": gt_docs,
            "retrieved_docs": pred_docs[:FINAL_TOP_K],
            "retrieved_chunk_types": chunk_types[:FINAL_TOP_K],
            "top1_score": top1_score,
            "retrieval_status": row.get("retrieval_status", ""),
            "coverage_reason": row.get("coverage_reason", ""),
        })

failure_df = pd.DataFrame(analysis_rows)
failure_df["question_short"] = failure_df["question"].apply(lambda x: _short(x, 70))
failure_df["gt_docs_short"] = failure_df["gt_docs"].apply(lambda x: _short(x, 80))
failure_df["retrieved_docs_short"] = failure_df["retrieved_docs"].apply(lambda x: _short(x, 95))
failure_df["retrieved_chunk_types_short"] = failure_df["retrieved_chunk_types"].apply(lambda x: _short(x, 70))
failure_df["top1_score_round"] = pd.to_numeric(failure_df["top1_score"], errors="coerce").round(4)

analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
failure_csv_path = analysis_dir / f"failure_analysis_p4_hwpx_{CORPUS_LIMIT}_sample{EVAL_SAMPLE_SIZE or 'all'}.csv"
failure_export = failure_df.copy()
for col in ["gt_docs", "retrieved_docs", "retrieved_chunk_types"]:
    failure_export[col] = failure_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
failure_export.to_csv(failure_csv_path, index=False, encoding="utf-8-sig")

print("analysis rows:", len(failure_df))
print("experiments:", sorted(failure_df["experiment_id"].unique()))
print("saved full failure analysis:", failure_csv_path)

_display_title("1) Hit/Miss count by experiment")
display(
    failure_df
    .groupby(["experiment_id", "hit_at_5"])
    .size()
    .reset_index(name="count")
    .sort_values(["experiment_id", "hit_at_5"])
)

_display_title("2) Miss count by type/difficulty/gt_doc_count")
display(
    failure_df[failure_df["hit_at_5"] == 0]
    .groupby(["experiment_id", "type", "difficulty", "gt_doc_count"])
    .size()
    .reset_index(name="miss_count")
    .sort_values(["experiment_id", "miss_count"], ascending=[True, False])
    .head(40)
)

_display_title("3) Miss count by retrieved chunk type mix")
chunk_mix = failure_df[failure_df["hit_at_5"] == 0].copy()
chunk_mix["retrieved_chunk_type_mix"] = chunk_mix["retrieved_chunk_types"].apply(lambda xs: " | ".join([x for x in xs if x]))
display(
    chunk_mix
    .groupby(["experiment_id", "retrieved_chunk_type_mix"])
    .size()
    .reset_index(name="miss_count")
    .sort_values(["experiment_id", "miss_count"], ascending=[True, False])
    .head(20)
)


analysis rows: 300
experiments: ['R0_p4_hwpx_690_v1_dense_top5', 'R1_p4_hwpx_690_v2_dense_rerank_top5', 'R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5']
saved full failure analysis: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/analysis/failure_analysis_p4_hwpx_690_sample100.csv

1) Hit/Miss count by experiment


,experiment_id,hit_at_5,count
0,R0_p4_hwpx_690_v1_dense_top5,0,14
1,R0_p4_hwpx_690_v1_dense_top5,1,86
2,R1_p4_hwpx_690_v2_dense_rerank_top5,0,5
3,R1_p4_hwpx_690_v2_dense_rerank_top5,1,95
4,R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5,0,1
5,R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5,1,99



2) Miss count by type/difficulty/gt_doc_count


,experiment_id,type,difficulty,gt_doc_count,miss_count
8,R0_p4_hwpx_690_v1_dense_top5,E,하,1,3
1,R0_p4_hwpx_690_v1_dense_top5,B,중,2,2
3,R0_p4_hwpx_690_v1_dense_top5,C,중,1,2
5,R0_p4_hwpx_690_v1_dense_top5,D,중,1,2
0,R0_p4_hwpx_690_v1_dense_top5,A,하,1,1
2,R0_p4_hwpx_690_v1_dense_top5,C,상,1,1
4,R0_p4_hwpx_690_v1_dense_top5,C,하,1,1
6,R0_p4_hwpx_690_v1_dense_top5,D,하,1,1
7,R0_p4_hwpx_690_v1_dense_top5,E,중,1,1
9,R1_p4_hwpx_690_v2_dense_rerank_top5,A,중,1,1



3) Miss count by retrieved chunk type mix


,experiment_id,retrieved_chunk_type_mix,miss_count
2,R0_p4_hwpx_690_v1_dense_top5,text | text | text | text | text,7
1,R0_p4_hwpx_690_v1_dense_top5,text | text | text | text,4
0,R0_p4_hwpx_690_v1_dense_top5,text | text | text,3
3,R1_p4_hwpx_690_v2_dense_rerank_top5,table,1
4,R1_p4_hwpx_690_v2_dense_rerank_top5,table | table | table | table | table,1
5,R1_p4_hwpx_690_v2_dense_rerank_top5,text | table | text,1
6,R1_p4_hwpx_690_v2_dense_rerank_top5,text | table | text | text,1
7,R1_p4_hwpx_690_v2_dense_rerank_top5,text | text | table | text | text,1
8,R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5,text | text | table | text | table,1


In [16]:
# ===== Compact R0/R1/R2 hit pattern summary =====
pivot_cols = ["source_eval_file", "id", "question", "type", "difficulty", "gt_doc_count"]
hit_pivot = failure_df.pivot_table(
    index=pivot_cols,
    columns="experiment_id",
    values="hit_at_5",
    aggfunc="max",
    fill_value=0,
).reset_index()

experiment_cols = [c for c in hit_pivot.columns if str(c).startswith("R")]
alias_map = {col: str(col).split("_", 1)[0] for col in experiment_cols}
print("experiment aliases:")
for col in experiment_cols:
    print(f"  {alias_map[col]} = {col}")

hit_pivot["question_short"] = hit_pivot["question"].apply(lambda x: _short(x, 70))

if len(experiment_cols) >= 2:
    base_col = experiment_cols[0]
    best_col = experiment_cols[-1]
    hit_pivot["pattern"] = hit_pivot[experiment_cols].astype(str).agg("".join, axis=1)
    view_cols = ["source_eval_file", "id", "type", "difficulty", "gt_doc_count", "question_short"] + experiment_cols

    _display_title("4) Hit pattern summary")
    display(hit_pivot["pattern"].value_counts().rename_axis("hit_pattern").reset_index(name="questions"))

    def _compact_pattern_view(df, max_rows=12):
        out = df[view_cols].copy().rename(columns=alias_map)
        return out.head(max_rows)

    _display_title("5) Final experiment improved over baseline")
    improved = hit_pivot[(hit_pivot[base_col] == 0) & (hit_pivot[best_col] == 1)]
    display(_compact_pattern_view(improved, 12))

    _display_title("6) Baseline hit but final experiment missed")
    regressed = hit_pivot[(hit_pivot[base_col] == 1) & (hit_pivot[best_col] == 0)]
    display(_compact_pattern_view(regressed, 12))

    _display_title("7) All experiments missed")
    all_missed = hit_pivot[hit_pivot[experiment_cols].sum(axis=1) == 0]
    display(_compact_pattern_view(all_missed, 12))
else:
    print("Not enough experiment columns to compare.")


experiment aliases:
  R0 = R0_p4_hwpx_690_v1_dense_top5
  R1 = R1_p4_hwpx_690_v2_dense_rerank_top5
  R2 = R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5

4) Hit pattern summary


,hit_pattern,questions
0,111,84
1,011,11
2,001,2
3,101,2
4,000,1



5) Final experiment improved over baseline


experiment_id,source_eval_file,id,type,difficulty,gt_doc_count,question_short,R0,R1,R2
9,eval_batch_02.csv,Q040,E,중,1,고대 차세대 ERP 말고 이번 학사 포털 플젝에서 메인으로 잡고 있는 레인지랑 타겟 베네핏이 어케 되는지 RFP 랩업해...,0,0,1
11,eval_batch_03.csv,Q056,C,중,1,"그렇다면 이 예산 하에서, 언급하신 주요 대학 구축 시스템 중 내부 기초 데이터 기반의 경영 구조 관련 부분은 어느 부...",0,1,1
20,eval_batch_04.csv,Q078,D,중,1,나노종합기술원이 발주한 23.4억여 원의 '스마트 팹 서비스 설비온라인 사업' 관련 제안서에 규정된 개발진의 야근 및 ...,0,1,1
21,eval_batch_04.csv,Q079,E,하,1,쑤자원공샤에서 용인 첨단 빤도체 구까산단으루 용수 공갑하는 씨스템 F/S 사옵으이 토탈 에샤는 얼마입니꺄?,0,1,1
22,eval_batch_04.csv,Q080,E,하,1,그래드코뤼아레져가 올해 스톼트하는 그룹વે어 메신져 통함 쁠젝의 젼체 뻐젯은 정확히 어떻게 잡혔능가요.,0,1,1
48,eval_batch_14.csv,Q278,D,중,1,"을지대학교 비교과시스템 개발 용역 지침에서, 학생 심리상담 예약 연동 모듈을 개발할 때 우울증 고위험군 재학생들에게 대...",0,1,1
65,eval_batch_18.csv,Q357,D,하,1,"을지대학교 비교과시스템 개발 지시서에서, 마일리지 성적이 우수한 상위 동아리 학생 대표 5명에게 지급해야 하는 해외 유...",0,1,1
67,eval_batch_19.csv,Q363,A,하,1,광주과학기술원(GIST)의 통합연구비관리망은 외부의 산업통상자원부 산하 어떤 메가 국가 결제 시스템과 연계하는 것을 목...,0,1,1
74,eval_batch_20.csv,Q392,B,중,2,케빈랩 평택시 관내 AI 영상감시망과 한국교육과정평가원의 국가교육과정 NCIC 운영망은 모두 '비상시(재난 및 장애시)...,0,0,1
87,eval_batch_23.csv,Q456,C,중,1,그렇다면 부산 쪽의 또 다른 공공기관인 부산관광공사의 경영정보망 개선 용역을 봅시다. 내부 임직원용 그룹웨어를 통합 구...,0,1,1



6) Baseline hit but final experiment missed


experiment_id,source_eval_file,id,type,difficulty,gt_doc_count,question_short,R0,R1,R2



7) All experiments missed


experiment_id,source_eval_file,id,type,difficulty,gt_doc_count,question_short,R0,R1,R2
45,eval_batch_12.csv,Q239,E,하,1,그뎌기요 보홈개뱔원애서 하는 실쏜보홈 쳥구 전싼화 사옵 그 예산 쳥액이 맹시적으로 어뜨케 표시대어 잇서요?,0,0,0


In [17]:
# ===== Compact failure case list =====
# Pick a CASE_ID from this compact table, then run the next cell to inspect it in plain text.

DETAIL_ROWS = 25
FAIL_CASES = failure_df[failure_df["hit_at_5"] == 0].copy().reset_index(drop=True)
FAIL_CASES["case_id"] = FAIL_CASES.index
FAIL_CASES = FAIL_CASES.sort_values(["experiment_id", "top1_score_round"], ascending=[True, True], na_position="last").reset_index(drop=True)
FAIL_CASES["case_id"] = FAIL_CASES.index

compact_case_cols = [
    "case_id",
    "experiment_id",
    "source_eval_file",
    "id",
    "type",
    "difficulty",
    "gt_doc_count",
    "top1_score_round",
    "question_short",
]

print("full detail csv:", failure_csv_path)
print("total miss cases:", len(FAIL_CASES))
display(FAIL_CASES[compact_case_cols].head(DETAIL_ROWS))


full detail csv: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/analysis/failure_analysis_p4_hwpx_690_sample100.csv
total miss cases: 20


,case_id,experiment_id,source_eval_file,id,type,difficulty,gt_doc_count,top1_score_round,question_short
0,0,R0_p4_hwpx_690_v1_dense_top5,eval_batch_04.csv,Q080,E,하,1,0.1618,그래드코뤼아레져가 올해 스톼트하는 그룹વે어 메신져 통함 쁠젝의 젼체 뻐젯은 정확히 어떻게 잡혔능가요.
1,1,R0_p4_hwpx_690_v1_dense_top5,eval_batch_04.csv,Q079,E,하,1,0.3298,쑤자원공샤에서 용인 첨단 빤도체 구까산단으루 용수 공갑하는 씨스템 F/S 사옵으이 토탈 에샤는 얼마입니꺄?
2,2,R0_p4_hwpx_690_v1_dense_top5,eval_batch_02.csv,Q040,E,중,1,0.4001,고대 차세대 ERP 말고 이번 학사 포털 플젝에서 메인으로 잡고 있는 레인지랑 타겟 베네핏이 어케 되는지 RFP 랩업해...
3,3,R0_p4_hwpx_690_v1_dense_top5,eval_batch_12.csv,Q239,E,하,1,0.4051,그뎌기요 보홈개뱔원애서 하는 실쏜보홈 쳥구 전싼화 사옵 그 예산 쳥액이 맹시적으로 어뜨케 표시대어 잇서요?
4,4,R0_p4_hwpx_690_v1_dense_top5,eval_batch_25.csv,Q491,B,중,2,0.4374,고양도시관리공사 다목적구장 예매망과 부산국제영화제의 행사 예약 포털은 시민들의 예약 트래픽을 처리하는 시스템입니다. 이...
5,5,R0_p4_hwpx_690_v1_dense_top5,eval_batch_14.csv,Q278,D,중,1,0.4499,"을지대학교 비교과시스템 개발 용역 지침에서, 학생 심리상담 예약 연동 모듈을 개발할 때 우울증 고위험군 재학생들에게 대..."
6,6,R0_p4_hwpx_690_v1_dense_top5,eval_batch_04.csv,Q078,D,중,1,0.4510,나노종합기술원이 발주한 23.4억여 원의 '스마트 팹 서비스 설비온라인 사업' 관련 제안서에 규정된 개발진의 야근 및 ...
7,7,R0_p4_hwpx_690_v1_dense_top5,eval_batch_18.csv,Q357,D,하,1,0.4528,"을지대학교 비교과시스템 개발 지시서에서, 마일리지 성적이 우수한 상위 동아리 학생 대표 5명에게 지급해야 하는 해외 유..."
8,8,R0_p4_hwpx_690_v1_dense_top5,eval_batch_25.csv,Q496,C,상,1,0.4773,그렇군요. 비슷한 맥락으로 경영 정보를 현대화하는 부산관광공사 시스템 개선을 여쭤보죠. 과거엔 모바일 지원이 안 돼서 ...
9,9,R0_p4_hwpx_690_v1_dense_top5,eval_batch_20.csv,Q392,B,중,2,0.5092,케빈랩 평택시 관내 AI 영상감시망과 한국교육과정평가원의 국가교육과정 NCIC 운영망은 모두 '비상시(재난 및 장애시)...


In [33]:
# ===== Failure CASE_ID detail viewer =====
# Change CASE_IDS and rerun only this cell.
# Examples: CASE_IDS = [0] or CASE_IDS = [0, 1, 2, 3, 4]

CASE_IDS = [0, 1, 2, 3, 4]
MAX_TEXT_CHARS = 900


def show_failure_case(case_id: int):
    if "FAIL_CASES" not in globals() or FAIL_CASES.empty:
        print("FAIL_CASES is empty. Run the compact failure case list cell first.")
        return
    matches = FAIL_CASES[FAIL_CASES["case_id"] == case_id]
    if matches.empty:
        print("Unknown CASE_ID:", case_id)
        return
    miss = matches.iloc[0]
    print("=" * 120)
    print(f"CASE_ID: {case_id}")
    print(f"EXPERIMENT: {miss['experiment_id']}")
    print(f"EVAL: {miss['source_eval_file']} / {miss['id']}")
    print(f"TYPE: {miss['type']} | DIFFICULTY: {miss['difficulty']} | GT_DOC_COUNT: {miss['gt_doc_count']} | TOP1: {miss.get('top1_score_round')}")
    print()
    print("QUESTION:")
    print(miss["question"])
    print()
    print("GT_DOCS:")
    for doc in miss["gt_docs"]:
        print("-", doc)
    print()
    print("RETRIEVED_DOCS:")
    for doc, chunk_type in zip(miss["retrieved_docs"], miss["retrieved_chunk_types"]):
        print(f"- [{chunk_type}] {doc}")

    pred_path = None
    for exp, path in _prediction_paths:
        if getattr(exp, "experiment_id", str(exp)) == miss["experiment_id"]:
            pred_path = Path(path)
            break
    if pred_path is None:
        return

    pred_df = _load_prediction_jsonl(pred_path)
    recs = pred_df[(pred_df["source_eval_file"].astype(str) == str(miss["source_eval_file"])) & (pred_df["id"].astype(str) == str(miss["id"]))]
    if recs.empty:
        return

    print()
    print("RETRIEVED CONTEXTS:")
    contexts = recs.iloc[0].get("retrieved_contexts", [])
    for ctx in _safe_contexts(contexts)[:FINAL_TOP_K]:
        text = str(ctx.get("text", "")).replace("\n", " ")
        if len(text) > MAX_TEXT_CHARS:
            text = text[:MAX_TEXT_CHARS].rstrip() + " ..."
        meta = ctx.get("metadata") or {}
        print("-" * 100)
        print(f"rank={ctx.get('rank')} score={ctx.get('score')} file={ctx.get('filename')}")
        print(f"chunk_type={meta.get('chunk_type')} chunk_id={ctx.get('chunk_id')}")
        print(text)


for case_id in CASE_IDS:
    show_failure_case(case_id)


CASE_ID: 0
EXPERIMENT: R0_p4_hwpx_690_v1_dense_top5
EVAL: eval_batch_04.csv / Q080
TYPE: E | DIFFICULTY: 하 | GT_DOC_COUNT: 1 | TOP1: 0.1618

QUESTION:
그래드코뤼아레져가 올해 스톼트하는 그룹વે어 메신져 통함 쁠젝의 젼체 뻐젯은 정확히 어떻게 잡혔능가요.

GT_DOCS:
- 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

RETRIEVED_DOCS:
- [text] 국가철도공단_철도산업정보센터 정보시스템 G-클라우드 1차 전환 용역.hwp
- [text] 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp
- [text] 서울에너지공사_그룹웨어시스템 고도화 용역.hwp
- [text] 국립공원공단_2024년 자원통합관리시스템 운영관리.hwp
- [text] 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp

RETRIEVED CONTEXTS:
----------------------------------------------------------------------------------------------------
rank=1 score=0.16183727979660034 file=국가철도공단_철도산업정보센터 정보시스템 G-클라우드 1차 전환 용역.hwp
chunk_type=text chunk_id=doc_6f4fab3c51ed_text_text_0061_part_001_0bae99e78d5e
[문서: 국가철도공단_철도산업정보센터 정보시스템 G-클라우드 1차 전환 용역.hwp | 사업명: 철도산업정보센터 정보시스템 G-클라우드 1차 전환 용역 | 발주기관: 국가철도공단 | 섹션: Ⅳ. 기타 | 유형: text] 그룹: 비밀번호는 정보보안담당관이 별도로 기록·관리하고 수시로 해당 계정에 접속하여 자정된 자료와 작업이력 확인 row_group: 비밀번호는 정보보안담당관이 별도로

## Diagnostic Deep Dive

Plain-text analysis for comparing P2/P4, reviewing failure and success cases, and deciding the next improvement direction. Run these cells after the existing evaluation and failure-analysis cells.


In [34]:
# ===== Plain-text diagnostic helpers =====
import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

from collections import Counter
import math

DIAG_CONTEXT_CHARS = 700
DIAG_CASE_LIMIT = 8


def _diag_line(char="-", n=110):
    print(char * n)


def _diag_title(title):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)


def _fmt_float(value, digits=4):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
        return f"{float(value):.{digits}f}"
    except Exception:
        return str(value)


def _clean_text(value, max_chars=120):
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    return text if len(text) <= max_chars else text[:max_chars].rstrip() + " ..."


def _norm_docs(values):
    return [_normalize_doc_name_for_analysis(v) for v in values if str(v).strip()]


def _missing_gold_docs(pred_docs, gt_docs):
    pred = set(_norm_docs(pred_docs[:FINAL_TOP_K]))
    missing = []
    for doc in gt_docs:
        if _normalize_doc_name_for_analysis(doc) not in pred:
            missing.append(doc)
    return missing


def _context_precision(pred_docs, gt_docs, k=5):
    pred_norm = _norm_docs(pred_docs[:k])
    gt_norm = set(_norm_docs(gt_docs))
    if not pred_norm:
        return 0.0
    return sum(1 for doc in pred_norm if doc in gt_norm) / len(pred_norm)


def _duplicate_doc_rate(pred_docs, k=5):
    docs = [d for d in pred_docs[:k] if str(d).strip()]
    if not docs:
        return 0.0
    return 1 - (len(set(_norm_docs(docs))) / len(docs))


def _get_exp_cols():
    if "experiment_cols" in globals() and experiment_cols:
        return list(experiment_cols)
    return sorted(failure_df["experiment_id"].unique())


def _exp_alias(exp_id):
    return str(exp_id).split("_", 1)[0]


def _get_pred_record(exp_id, source_eval_file, qid):
    if "_DIAG_PRED_RECORDS" not in globals():
        return None
    return _DIAG_PRED_RECORDS.get(exp_id, {}).get((str(source_eval_file), str(qid)))


_DIAG_PRED_RECORDS = {}
for exp, pred_path in _prediction_paths:
    exp_id = getattr(exp, "experiment_id", str(exp))
    pred_df = _load_prediction_jsonl(Path(pred_path))
    lookup = {}
    for _, row in pred_df.iterrows():
        lookup[(str(row.get("source_eval_file", "")), str(row.get("id", "")))] = row.to_dict()
    _DIAG_PRED_RECORDS[exp_id] = lookup

_DIAG_EXPERIMENTS = _get_exp_cols()
_DIAG_BASE_EXP = _DIAG_EXPERIMENTS[0]
_DIAG_FINAL_EXP = _DIAG_EXPERIMENTS[-1]
print("diagnostic experiments:")
for exp_id in _DIAG_EXPERIMENTS:
    print(f"  {_exp_alias(exp_id)} = {exp_id}")


diagnostic experiments:
  R0 = R0_p4_hwpx_690_v1_dense_top5
  R1 = R1_p4_hwpx_690_v2_dense_rerank_top5
  R2 = R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5


In [35]:
# ===== Metric supplements: ranking, precision, duplicates =====
_diag_title("A) Supplemental retrieval metrics")
print("These metrics help explain why nDCG / recall changed even when Hit@5 is high.")
print()
header = f"{'EXP':<4} {'top1_hit':>9} {'avg_rank*':>9} {'median_rank*':>12} {'context_precision@5':>20} {'duplicate_doc_rate@5':>21} {'multi_recall@5':>15}"
print(header)
print("-" * len(header))
for exp_id in _DIAG_EXPERIMENTS:
    part = failure_df[failure_df["experiment_id"] == exp_id].copy()
    top1_hit = (part["first_gold_rank"] == 1).mean()
    penalized_rank = part["first_gold_rank"].fillna(FINAL_TOP_K + 1).astype(float)
    context_precision = part.apply(lambda r: _context_precision(r["retrieved_docs"], r["gt_docs"], FINAL_TOP_K), axis=1).mean()
    duplicate_rate = part.apply(lambda r: _duplicate_doc_rate(r["retrieved_docs"], FINAL_TOP_K), axis=1).mean()
    multi = part[part["gt_doc_count"] > 1]
    multi_recall = multi["doc_recall_at_5"].mean() if len(multi) else float("nan")
    print(
        f"{_exp_alias(exp_id):<4} "
        f"{top1_hit:>9.3f} "
        f"{penalized_rank.mean():>9.3f} "
        f"{penalized_rank.median():>12.3f} "
        f"{context_precision:>20.3f} "
        f"{duplicate_rate:>21.3f} "
        f"{multi_recall:>15.3f}"
    )
print()
print("* avg_rank/median_rank use 6 as penalty for a miss outside top5.")

_diag_title("B) Chunk type mix in final experiment")
final_part = failure_df[failure_df["experiment_id"] == _DIAG_FINAL_EXP].copy()
for hit_value, label in [(1, "HIT"), (0, "MISS")]:
    subset = final_part[final_part["hit_at_5"] == hit_value]
    counter = Counter()
    for xs in subset["retrieved_chunk_types"]:
        counter.update([x or "unknown" for x in xs])
    print(f"{label}: questions={len(subset)} chunk_type_counts={dict(counter)}")



A) Supplemental retrieval metrics
These metrics help explain why nDCG / recall changed even when Hit@5 is high.

EXP   top1_hit avg_rank* median_rank*  context_precision@5  duplicate_doc_rate@5  multi_recall@5
------------------------------------------------------------------------------------------------
R0       0.760     1.890        1.000                0.483                 0.000           0.652
R1       0.910     1.300        1.000                0.769                 0.000           0.714
R2       0.930     1.140        1.000                0.822                 0.000           0.748

* avg_rank/median_rank use 6 as penalty for a miss outside top5.

B) Chunk type mix in final experiment
HIT: questions=99 chunk_type_counts={'text': 131, 'table': 37, 'fact_candidates': 2}
MISS: questions=1 chunk_type_counts={'text': 3, 'table': 2}


In [36]:
# ===== Case review: recovered, regressed, all-missed, and multi-doc loss =====
# Important: hit_pivot is used only to SELECT cases.
# GT docs are always reloaded from eval_df by (source_eval_file, id).
# Retrieved contexts are always reloaded from prediction JSONL records.

def _print_docs(title, docs, max_items=5):
    print(title)
    docs = list(docs or [])
    if not docs:
        print("  - none")
        return
    for doc in docs[:max_items]:
        print("  -", doc)


def _eval_case_lookup(source_eval_file, qid):
    source_eval_file = str(source_eval_file)
    qid = str(qid)
    matches = eval_df[
        (eval_df["source_eval_file"].astype(str) == source_eval_file)
        & (eval_df["id"].astype(str) == qid)
    ]
    if matches.empty:
        return None
    row = matches.iloc[0]
    gt_docs = row.get("ground_truth_doc_list", [])
    if not isinstance(gt_docs, list):
        gt_docs = []
    return {
        "source_eval_file": source_eval_file,
        "id": qid,
        "question": row.get("question", ""),
        "type": row.get("type", ""),
        "difficulty": row.get("difficulty", ""),
        "gt_docs": gt_docs,
        "gt_doc_count": len(gt_docs),
    }


def _case_from_any_row(row):
    source_eval_file = str(row.get("source_eval_file", ""))
    qid = str(row.get("id", ""))
    case = _eval_case_lookup(source_eval_file, qid)
    if case is None:
        print("WARNING: GT docs not found for", source_eval_file, qid)
        return {
            "source_eval_file": source_eval_file,
            "id": qid,
            "question": row.get("question", ""),
            "type": row.get("type", ""),
            "difficulty": row.get("difficulty", ""),
            "gt_docs": [],
            "gt_doc_count": row.get("gt_doc_count", ""),
        }
    return case


def _print_case_summary(row, show_all_exp=False, context_chars=DIAG_CONTEXT_CHARS):
    case = _case_from_any_row(row)
    source_eval_file = case["source_eval_file"]
    qid = case["id"]
    gt_docs = case["gt_docs"]

    print("\n" + "-" * 110)
    print(
        f"CASE: {source_eval_file} / {qid} | "
        f"type={case.get('type')} | difficulty={case.get('difficulty')} | gt_doc_count={case.get('gt_doc_count')}"
    )
    print("QUESTION:", _clean_text(case.get("question", ""), 260))
    _print_docs("GT_DOCS:", gt_docs, 8)

    if not gt_docs:
        print("WARNING: empty GT docs. Treat this as an analysis-data issue before interpreting the case.")

    exp_ids = _DIAG_EXPERIMENTS if show_all_exp else [_DIAG_FINAL_EXP]
    for exp_id in exp_ids:
        rec = _get_pred_record(exp_id, source_eval_file, qid)
        if not rec:
            print(f"\n[{_exp_alias(exp_id)}] WARNING: prediction record not found")
            continue
        contexts = _safe_contexts(rec.get("retrieved_contexts", []))
        pred_docs = _context_doc_names(contexts)
        missing = _missing_gold_docs(pred_docs, gt_docs)
        hit = _hit_any_at_k(pred_docs, gt_docs, FINAL_TOP_K)
        recall = _doc_recall_at_k(pred_docs, gt_docs, FINAL_TOP_K)
        first_rank = _rank_first_gold(pred_docs, gt_docs, FINAL_TOP_K)
        precision = _context_precision(pred_docs, gt_docs, FINAL_TOP_K)
        dup_rate = _duplicate_doc_rate(pred_docs, FINAL_TOP_K)

        print(
            f"\n[{_exp_alias(exp_id)}] "
            f"hit={hit} recall={_fmt_float(recall)} precision={_fmt_float(precision)} "
            f"first_gold_rank={first_rank} missing_gold={len(missing)} dup_doc_rate={_fmt_float(dup_rate)}"
        )
        _print_docs("retrieved_docs:", pred_docs, FINAL_TOP_K)
        if missing:
            _print_docs("missing_gold_docs:", missing, 8)
        print("contexts:")
        for ctx in contexts[:FINAL_TOP_K]:
            meta = ctx.get("metadata") or {}
            text = _clean_text(str(ctx.get("text", "")).replace("\n", " "), context_chars)
            print(f"  #{ctx.get('rank')} score={_fmt_float(ctx.get('score'))} type={meta.get('chunk_type')} file={ctx.get('filename')}")
            print("   ", text)


def _print_case_list(title, rows, show_all_exp, context_chars, limit=DIAG_CASE_LIMIT):
    _diag_title(title)
    print(f"count={len(rows)}")
    if len(rows) == 0:
        return
    for _, row in rows.head(limit).iterrows():
        _print_case_summary(row, show_all_exp=show_all_exp, context_chars=context_chars)


if "hit_pivot" not in globals():
    raise RuntimeError("hit_pivot is missing. Run the previous failure analysis cells first.")

recovered = hit_pivot[(hit_pivot[_DIAG_BASE_EXP] == 0) & (hit_pivot[_DIAG_FINAL_EXP] == 1)].copy()
_print_case_list("C) Cases recovered by final R2 over baseline R0", recovered, show_all_exp=True, context_chars=360)

final_missed = hit_pivot[hit_pivot[_DIAG_FINAL_EXP] == 0].copy()
_print_case_list("D) Cases missed by final R2", final_missed, show_all_exp=True, context_chars=500)

final_rows = failure_df[failure_df["experiment_id"] == _DIAG_FINAL_EXP].copy()
multi_loss = final_rows[(final_rows["gt_doc_count"] > 1) & (final_rows["doc_recall_at_5"] < 1)].copy()
multi_loss = multi_loss.sort_values(["doc_recall_at_5", "first_gold_rank"], ascending=[True, True], na_position="last")
_print_case_list("E) Multi-doc cases where final R2 did not retrieve all gold docs", multi_loss, show_all_exp=False, context_chars=420)



C) Cases recovered by final R2 over baseline R0
count=13

--------------------------------------------------------------------------------------------------------------
CASE: eval_batch_02.csv / Q040 | type=E | difficulty=중 | gt_doc_count=1
QUESTION: 고대 차세대 ERP 말고 이번 학사 포털 플젝에서 메인으로 잡고 있는 레인지랑 타겟 베네핏이 어케 되는지 RFP 랩업해서 간략히 요약좀 해줄래?
GT_DOCS:
  - 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf

[R0] hit=0 recall=0.0000 precision=0.0000 first_gold_rank=None missing_gold=1 dup_doc_rate=0.0000
retrieved_docs:
  - 한국농어촌공사_차세대 ERP 통합업무포털시스템 구축 용역.hwp
  - 국제사이버대학교_(재공고) 국제사이버대학교 차세대 학사·행정 시스템.hwp
  - 국제사이버대학교_국제사이버대학교 차세대 학사·행정 시스템 구축 사.hwp
missing_gold_docs:
  - 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
contexts:
  #1 score=0.4001 type=text file=한국농어촌공사_차세대 ERP 통합업무포털시스템 구축 용역.hwp
    [문서: 한국농어촌공사_차세대 ERP 통합업무포털시스템 구축 용역.hwp | 사업명: 차세대 ERP 통합업무포털시스템 구축 용역 | 발주기관: 한국농어촌공사 | 섹션: 문서 시작 | 유형: text] Level 2: 예산관리 | Level 3: 6 | Level 4: 29 Level 2: 구매자재 | Level 3: 5 | Level 4: 24 Level 2: 인사관리 | Level 3: 12 | Level 4: 59 Leve

In [37]:
# ===== Improvement direction memo from observed patterns =====
_diag_title("F) Improvement direction memo")
final_rows = failure_df[failure_df["experiment_id"] == _DIAG_FINAL_EXP].copy()
all_missed_count = len(final_rows[final_rows["hit_at_5"] == 0])
multi_loss_count = len(final_rows[(final_rows["gt_doc_count"] > 1) & (final_rows["doc_recall_at_5"] < 1)])

print(f"final_experiment={_DIAG_FINAL_EXP}")
print(f"all_missed_count={all_missed_count}")
print(f"multi_doc_partial_loss_count={multi_loss_count}")
print()
print("Recommended next checks:")
print("1. If all-missed questions contain noisy spellings or phonetic Korean, add query normalization/rewrite before retrieval.")
print("2. If multi-doc partial loss remains, evaluate top_k=8 or top_k=10 for final contexts, or use doc-level grouping before final selection.")
print("3. If table chunks dominate wrong hits, reduce noisy table fields such as repeated row_group labels or add table_type filtering.")
print("4. Compare whether the gold document is present in dense_top30 or BM25_top30 before reranking. This separates candidate-generation failure from reranker failure.")
print("5. Keep R1 as a practical default if latency matters; use R2 as fallback for low-confidence or multi-doc queries.")



F) Improvement direction memo
final_experiment=R2_p4_hwpx_690_v2_hybrid_rrf_rerank_top5
all_missed_count=1
multi_doc_partial_loss_count=21

Recommended next checks:
1. If all-missed questions contain noisy spellings or phonetic Korean, add query normalization/rewrite before retrieval.
2. If multi-doc partial loss remains, evaluate top_k=8 or top_k=10 for final contexts, or use doc-level grouping before final selection.
3. If table chunks dominate wrong hits, reduce noisy table fields such as repeated row_group labels or add table_type filtering.
4. Compare whether the gold document is present in dense_top30 or BM25_top30 before reranking. This separates candidate-generation failure from reranker failure.
5. Keep R1 as a practical default if latency matters; use R2 as fallback for low-confidence or multi-doc queries.


## Candidate Stage Analysis

Checks where gold documents disappear for final failures and multi-doc partial losses. This does not rebuild embeddings or Chroma; it reuses the current runtime collections/BM25 objects.


In [32]:
# ===== Candidate-stage analysis: dense top30 vs BM25 top30 vs RRF vs final top5 =====
# Run after cells 17-24. Expected time on live runtime: usually 1-7 minutes for sample100.

CANDIDATE_CASE_LIMIT = 40
CANDIDATE_DENSE_TOP_K = 30
CANDIDATE_BM25_TOP_K = 30
CANDIDATE_RRF_TOP_K = 30


def _candidate_gold_stats(items, gt_docs, k):
    docs = _context_doc_names(items[:k])
    gt_norm = set(_norm_docs(gt_docs))
    pred_norm = _norm_docs(docs)
    hit_docs = [doc for doc in gt_docs if _normalize_doc_name_for_analysis(doc) in set(pred_norm)]
    missing_docs = [doc for doc in gt_docs if _normalize_doc_name_for_analysis(doc) not in set(pred_norm)]
    first_rank = None
    for idx, doc in enumerate(pred_norm, start=1):
        if doc in gt_norm:
            first_rank = idx
            break
    recall = len(hit_docs) / len(gt_docs) if gt_docs else 0.0
    return {
        "hit_any": int(bool(hit_docs)),
        "recall": recall,
        "first_rank": first_rank,
        "hit_docs": hit_docs,
        "missing_docs": missing_docs,
    }


def _candidate_docs(items, k=5):
    docs = _context_doc_names(items[:k])
    types = _context_chunk_types(items[:k])
    return [f"[{t}] {d}" for d, t in zip(docs, types)]


def _analyze_candidate_stage_for_case(source_eval_file, qid):
    case = _eval_case_lookup(source_eval_file, qid)
    if case is None:
        return None
    question = case["question"]
    gt_docs = case["gt_docs"]

    dense_items, _, dense_ms = dense_candidates(collection_v2, question, n_results=CANDIDATE_DENSE_TOP_K)
    bm25_items = bm25_v2.search(question, top_k=CANDIDATE_BM25_TOP_K) if "bm25_v2" in globals() and bm25_v2 is not None else []
    rrf_items = rrf_fuse([dense_items, bm25_items], rrf_k=RRF_K, top_k=CANDIDATE_RRF_TOP_K)

    final_rec = _get_pred_record(_DIAG_FINAL_EXP, source_eval_file, qid)
    final_items = _safe_contexts(final_rec.get("retrieved_contexts", [])) if final_rec else []

    return {
        "case": case,
        "dense_ms": dense_ms,
        "dense": _candidate_gold_stats(dense_items, gt_docs, CANDIDATE_DENSE_TOP_K),
        "bm25": _candidate_gold_stats(bm25_items, gt_docs, CANDIDATE_BM25_TOP_K),
        "rrf": _candidate_gold_stats(rrf_items, gt_docs, CANDIDATE_RRF_TOP_K),
        "final": _candidate_gold_stats(final_items, gt_docs, FINAL_TOP_K),
        "dense_docs": _candidate_docs(dense_items, 5),
        "bm25_docs": _candidate_docs(bm25_items, 5),
        "rrf_docs": _candidate_docs(rrf_items, 5),
        "final_docs": _candidate_docs(final_items, 5),
    }


def _candidate_failure_reason(result):
    if result is None:
        return "analysis_lookup_failed"
    dense_hit = result["dense"]["hit_any"]
    bm25_hit = result["bm25"]["hit_any"]
    rrf_hit = result["rrf"]["hit_any"]
    final_hit = result["final"]["hit_any"]
    if not dense_hit and not bm25_hit:
        return "candidate_generation_failed"
    if (dense_hit or bm25_hit) and not rrf_hit:
        return "rrf_fusion_lost_gold"
    if rrf_hit and not final_hit:
        return "rerank_or_final_selection_lost_gold"
    if final_hit and result["final"]["recall"] < 1:
        return "partial_multi_doc_loss"
    return "ok"


def _print_candidate_stage_result(result, max_docs=5):
    case = result["case"]
    print("\n" + "=" * 120)
    print(f"CASE: {case['source_eval_file']} / {case['id']} | type={case['type']} | difficulty={case['difficulty']} | gt_doc_count={case['gt_doc_count']}")
    print("QUESTION:", _clean_text(case["question"], 240))
    _print_docs("GT_DOCS:", case["gt_docs"], 8)
    print("reason:", _candidate_failure_reason(result))
    print()
    header = f"{'stage':<8} {'hit':>3} {'recall':>7} {'first_rank':>10} {'missing_gold':>12}"
    print(header)
    print("-" * len(header))
    for stage in ["dense", "bm25", "rrf", "final"]:
        stat = result[stage]
        print(f"{stage:<8} {stat['hit_any']:>3} {stat['recall']:>7.3f} {str(stat['first_rank']):>10} {len(stat['missing_docs']):>12}")
    for stage in ["dense", "bm25", "rrf", "final"]:
        print(f"\n{stage.upper()} top docs:")
        docs = result[f"{stage}_docs"]
        if not docs:
            print("  - none")
        for doc in docs[:max_docs]:
            print("  -", doc)


_diag_title("G) Candidate-stage summary for final misses and multi-doc partial losses")
if "hit_pivot" not in globals() or "failure_df" not in globals():
    raise RuntimeError("Run failure analysis cells first: 17, 18, 22, 23, 24.")

final_missed_keys = []
if "hit_pivot" in globals():
    final_missed = hit_pivot[hit_pivot[_DIAG_FINAL_EXP] == 0].copy()
    final_missed_keys = [(str(r["source_eval_file"]), str(r["id"])) for _, r in final_missed.iterrows()]

final_rows = failure_df[failure_df["experiment_id"] == _DIAG_FINAL_EXP].copy()
multi_loss = final_rows[(final_rows["gt_doc_count"] > 1) & (final_rows["doc_recall_at_5"] < 1)].copy()
multi_loss_keys = [(str(r["source_eval_file"]), str(r["id"])) for _, r in multi_loss.iterrows()]

candidate_keys = []
for key in final_missed_keys + multi_loss_keys:
    if key not in candidate_keys:
        candidate_keys.append(key)
candidate_keys = candidate_keys[:CANDIDATE_CASE_LIMIT]
print("final_missed_cases:", len(final_missed_keys))
print("multi_doc_partial_loss_cases:", len(multi_loss_keys))
print("candidate_stage_cases_to_check:", len(candidate_keys))

candidate_results = []
for source_eval_file, qid in candidate_keys:
    result = _analyze_candidate_stage_for_case(source_eval_file, qid)
    if result is None:
        continue
    reason = _candidate_failure_reason(result)
    row = {
        "source_eval_file": source_eval_file,
        "id": qid,
        "type": result["case"]["type"],
        "difficulty": result["case"]["difficulty"],
        "gt_doc_count": result["case"]["gt_doc_count"],
        "question_short": _clean_text(result["case"]["question"], 80),
        "reason": reason,
    }
    for stage in ["dense", "bm25", "rrf", "final"]:
        row[f"{stage}_hit"] = result[stage]["hit_any"]
        row[f"{stage}_recall"] = result[stage]["recall"]
        row[f"{stage}_first_rank"] = result[stage]["first_rank"]
        row[f"{stage}_missing_count"] = len(result[stage]["missing_docs"])
    candidate_results.append((row, result))

candidate_stage_df = pd.DataFrame([x[0] for x in candidate_results])
candidate_stage_path = OUTPUT_DIR / "analysis" / f"candidate_stage_p4_hwpx_{CORPUS_LIMIT}_sample{EVAL_SAMPLE_SIZE or 'all'}.csv"
candidate_stage_df.to_csv(candidate_stage_path, index=False, encoding="utf-8-sig")
print("saved candidate stage csv:", candidate_stage_path)

print("\nReason counts:")
if not candidate_stage_df.empty:
    print(candidate_stage_df["reason"].value_counts().to_string())
else:
    print("none")

print("\nCompact cases:")
if not candidate_stage_df.empty:
    for _, row in candidate_stage_df.head(20).iterrows():
        print(
            f"- {row['source_eval_file']} / {row['id']} | reason={row['reason']} | "
            f"dense={row['dense_recall']:.2f} bm25={row['bm25_recall']:.2f} rrf={row['rrf_recall']:.2f} final={row['final_recall']:.2f} | "
            f"{row['question_short']}"
        )

_diag_title("H) Candidate-stage detailed examples")
for row, result in candidate_results[: min(8, len(candidate_results))]:
    _print_candidate_stage_result(result, max_docs=5)



G) Candidate-stage summary for final misses and multi-doc partial losses
final_missed_cases: 1
multi_doc_partial_loss_cases: 21
candidate_stage_cases_to_check: 22
saved candidate stage csv: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_eval_p4_hwpx/analysis/candidate_stage_p4_hwpx_690_sample100.csv

Reason counts:
reason
partial_multi_doc_loss         21
candidate_generation_failed     1

Compact cases:
- eval_batch_12.csv / Q239 | reason=candidate_generation_failed | dense=0.00 bm25=0.00 rrf=0.00 final=0.00 | 그뎌기요 보홈개뱔원애서 하는 실쏜보홈 쳥구 전싼화 사옵 그 예산 쳥액이 맹시적으로 어뜨케 표시대어 잇서요?
- eval_batch_01.csv / Q010 | reason=partial_multi_doc_loss | dense=0.50 bm25=0.50 rrf=0.50 final=0.50 | 국립중앙의료원의 차세대 응급의료 상황관리시스템 사업과 한국원자력연구원의 선량평가시스템 고도화 사업 모두 시스템 고도화 및 구축을 목적으로 합니다 ...
- eval_batch_02.csv / Q031 | reason=partial_multi_doc_loss | dense=0.50 bm25=0.50 rrf=0.50 final=0.50 | 국립중앙의료원의 위탁용역과 그랜드코리아레저(주)의 그룹웨어 구축 용역에서 각각 핵심으로 다루는 정보시스템의 업무 목적과 범위를 비교해 주십시오.
- eval_batch_02.csv / Q034 | rea